In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.6 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [36]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-03-17 10:02:06,456] A new study created in memory with name: no-name-89a15aba-af04-4a1a-add9-15acbb59634c
[I 2026-03-17 10:02:06,935] Trial 0 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 80, 'max_depth': 9}. Best is trial 0 with value: 0.7597765363128491.
[I 2026-03-17 10:02:08,109] Trial 1 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 197, 'max_depth': 18}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-03-17 10:02:09,144] Trial 2 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 187, 'max_depth': 4}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-03-17 10:02:10,294] Trial 3 finished with value: 0.7560521415270017 and parameters: {'n_estimators': 200, 'max_depth': 6}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-03-17 10:02:11,397] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 185, 'max_depth': 15}. Best is trial 1 with value: 0.77653631

In [37]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279331
Best hyperparameters: {'n_estimators': 73, 'max_depth': 19}


In [38]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76


## RandomizedSearchCV in Optuna

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [26]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-03-17 09:59:22,679] A new study created in memory with name: no-name-b7833ebb-8b13-46cd-90b8-eeb48c1abb66
[I 2026-03-17 09:59:25,594] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 185, 'max_depth': 15}. Best is trial 0 with value: 0.7672253258845437.
[I 2026-03-17 09:59:26,928] Trial 1 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 77, 'max_depth': 17}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-03-17 09:59:28,142] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 70, 'max_depth': 11}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-03-17 09:59:29,036] Trial 3 finished with value: 0.756052141527002 and parameters: {'n_estimators': 93, 'max_depth': 3}. Best is trial 1 with value: 0.7746741154562384.
[I 2026-03-17 09:59:29,980] Trial 4 finished with value: 0.7783985102420857 and parameters: {'n_estimators': 106, 'max_depth': 15}. Best is trial 4 with value: 0.778398510

In [27]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 117, 'max_depth': 15}


In [28]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


# GridSearchCV in Optuna

In [29]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [30]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-03-17 10:01:17,353] A new study created in memory with name: no-name-d5d778d7-e650-4b5e-a8d8-041c6e94aaf9
[I 2026-03-17 10:01:18,435] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-03-17 10:01:20,358] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-03-17 10:01:20,928] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-17 10:01:22,013] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-17 10:01:23,293] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [31]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [32]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [39]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [40]:
# 1. Optimization History
plot_optimization_history(study).show()

In [41]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [42]:
# 3. Slice Plot
plot_slice(study).show()

In [43]:
# 4. Contour Plot
plot_contour(study).show()

In [44]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [45]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [53]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [54]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-03-17 10:26:04,966] A new study created in memory with name: no-name-5d4f00fd-8b99-417b-9fa7-4f99c53fcf9e
[I 2026-03-17 10:26:05,320] Trial 0 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 95.86423706827888, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 0 with value: 0.7858472998137801.
[I 2026-03-17 10:26:05,641] Trial 1 finished with value: 0.7597765363128491 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 53, 'learning_rate': 0.041605820491449896, 'max_depth': 3, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.7858472998137801.
[I 2026-03-17 10:26:06,465] Trial 2 finished with value: 0.7560521415270017 and parameters: {'classifier': 'RandomForest', 'n_estimators': 153, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 8, 'bootstrap': True}. Best is trial 0 with value: 0.7858472998137801.
[I 2026-03-17 10:26:06,498] Trial 3 finished with value: 0.6927374301675977 and paramete

In [55]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11197499489725579, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [57]:
from sklearn.metrics import accuracy_score

best_params = study.best_trial.params.copy()
best_params.pop("classifier")

# best_model = RandomForestClassifier(**best_params, random_state=42)
best_model = SVC(**best_params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


In [58]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.785847,2026-03-17 10:26:04.968532,2026-03-17 10:26:05.320345,0 days 00:00:00.351813,95.864237,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.759777,2026-03-17 10:26:05.321350,2026-03-17 10:26:05.641682,0 days 00:00:00.320332,NaN,NaN,GradientBoosting,NaN,NaN,0.041606,3.0,1.0,2.0,53.0,COMPLETE
2,2,0.756052,2026-03-17 10:26:05.642839,2026-03-17 10:26:06.465842,0 days 00:00:00.823003,NaN,True,RandomForest,NaN,NaN,NaN,7.0,8.0,3.0,153.0,COMPLETE
3,3,0.692737,2026-03-17 10:26:06.467045,2026-03-17 10:26:06.498047,0 days 00:00:00.031002,13.954771,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.763501,2026-03-17 10:26:06.499084,2026-03-17 10:26:06.986345,0 days 00:00:00.487261,NaN,False,RandomForest,NaN,NaN,NaN,15.0,3.0,4.0,94.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.709497,2026-03-17 10:26:44.517937,2026-03-17 10:26:44.550881,0 days 00:00:00.032944,0.116640,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2026-03-17 10:26:44.552829,2026-03-17 10:26:44.584702,0 days 00:00:00.031873,0.208179,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.698324,2026-03-17 10:26:44.585937,2026-03-17 10:26:44.625101,0 days 00:00:00.039164,26.022871,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.789572,2026-03-17 10:26:44.626093,2026-03-17 10:26:44.657397,0 days 00:00:00.031304,0.142261,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [59]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,80
GradientBoosting,10
RandomForest,10


In [60]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.751397
RandomForest,0.765922
SVM,0.776094


In [61]:
# 1. Optimization History
plot_optimization_history(study).show()

In [62]:
# 3. Slice Plot
plot_slice(study).show()

In [63]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()